# Reliability testing on dataset

In [ ]:
cd "/home/simon/Documents/Zwischen Wörtern und Pixeln/"

In [ ]:
#import custom Config
from Config import cfg

import pandas as pd
from pathlib import Path

## Get dataset

In [ ]:
#define acceptable file types
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}

#define function to collect image paths
def collect_image_paths(input_path: str) -> list[str]:
    """Accept a single image file or a directory (non-recursive)."""
    p = Path(input_path)
    
    if p.is_file():
        if p.suffix.lower() not in IMAGE_EXTENSIONS:
            raise ValueError(f"Unsupported file extension: {p.suffix}")
        return [str(p)]
    
    elif p.is_dir():
        paths = sorted(
            str(f) for f in p.iterdir()
            if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
        )
        if not paths:
            raise FileNotFoundError(f"No images found in directory: {input_path}")
        return paths
    
    else:
        raise FileNotFoundError(f"Input path does not exist: {input_path}")

In [ ]:
images = collect_image_paths("Sample_Dataset/Reichel/images_with_faces")

In [ ]:
#read in dataset
df_dataset = pd.DataFrame({"filename": images})

#verify
df_dataset.head()

## Fix filename for correct absolute path

We can use `Series.replace()` or `str.replace()`. <br>
We are going to use `str.replace()` as it is more efficient and faster (note: does not really matter for a small dataset like this).

In [ ]:
#fix filename
df_dataset["filename"] = df_dataset["filename"].str.replace(
    "Sample_Dataset/",
    "/home/simon/Documents/Zwischen Wörtern und Pixeln/Sample_Dataset/",
    regex = False
)

#verify
df_dataset.head()

## Get sample

In [ ]:
#get 200 random cases
df_sample = df_dataset.sample(n = 100, #n
                      random_state = cfg.random_seed #seed for reproducibility
                     )

#verify
df_sample.info()

## Copy images

In [ ]:
#copy all images into a separate folder
import shutil
from pathlib import Path

#set destination
destination = Path("Sample_Dataset/Reichel/reliability")
destination.mkdir(parents = True, exist_ok = True)

#copy files
for file in df_sample["filename"]:
    shutil.copy2(file, destination)

## Save to csv

In [ ]:
df_sample.to_csv("Sample_Dataset/Reichel/reliability.csv",
                 encoding = "UTF-8",
                 index = False
                )

<br><br><br><br><br><br>

## Import coded and classifier file for reliability test

coded file = sample manually coded by human coder <br>
classifier file = sample coded by ML model

In [ ]:
#read data
df_classifier = pd.read_csv("Sample_Dataset/Reichel/classifier_results_fp32.csv")

df_human = pd.read_csv("Sample_Dataset/Reichel/reliability_coded.csv")

## Preprocessing

In [ ]:
#change column so they match
df_classifier = df_classifier.rename(columns = {"image_path": "filename"})

In [ ]:
#change filename so they match
df_human["filename"] = df_human["filename"].str.replace(
    "images_with_faces",
    "reliability",
    regex = False
)

df_classifier["filename"] = df_classifier["filename"].str.replace(
    "Sample_Dataset/",
    "/home/simon/Documents/Zwischen Wörtern und Pixeln/Sample_Dataset/",
    regex = False
)

In [ ]:
#merge dataframes
df_combined = pd.merge(df_classifier,
                       df_human,
                       on = "filename"
                      )

In [ ]:
df_combined.head()

In [ ]:
#drop unnecessary columns
df_combined = df_combined.drop(["Optimistic", "Pessimistic", "Hostile", "Neutral"
                               ], axis = 1)

In [ ]:
#recode to integers
df_combined = df_combined.replace(
    ["Optimistic",
    "Pessimistic",
    "Hostile",
    "Neutral"],
    [1,
    2,
    3,
    4]
)

In [ ]:
#recode Unknowns to Zero
df_combined = df_combined.replace(
    ["Unknown"],
    [0]
)

#remove empty rows should they exist
df_combined = df_combined[
    df_combined["prediction"].notna() &
    (df_combined["prediction"] != 0) &
    (df_combined["prediction"].astype(str).str.strip() != "")
]

#remove empty rows should they exist
df_combined = df_combined[
    df_combined["code_human"].notna() &
    (df_combined["code_human"] != 0) &
    (df_combined["code_human"].astype(str).str.strip() != "")
]


In [ ]:
df_combined["prediction"] = df_combined["prediction"].astype(int)
df_combined["code_human"] = df_combined["code_human"].astype(int)

In [ ]:
print(len(df_combined))

In [ ]:
df_combined.head()

## Calculate metrics (accuracy & F1)

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

#define truths
y_true = df_combined["code_human"] #truth
y_pred = df_combined["prediction"] #estimate

#confusion matrix
cm = confusion_matrix(y_true, y_pred)

#metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average = "weighted")
recall = recall_score(y_true, y_pred, average = "weighted")
f1 = f1_score(y_true, y_pred, average = "weighted")

print("Confusion Matrix:")
print(cm)

#classification report
print(classification_report(y_true, y_pred))